In [1]:
import pandas as pd
import numpy as np

# loading Cleaned Datasets

In [2]:
cleaned_data_path = '../data/processed/'


def load_dataset(path):
    df = pd.read_csv(f'{cleaned_data_path}{path}')
    return df

# Load all raw CSV files
account_profiles_df_processed = load_dataset("account_profiles_clean.csv")
fraud_patterns_df_processed = load_dataset("fraud_patterns_clean.csv")
network_edges_df_processed = load_dataset("network_edges_clean.csv")
time_series_stats_df_processed = load_dataset("time_series_stats_clean.csv")
transactions_df_processed = load_dataset("transactions_clean.csv")

print("All cleaned CSV files loaded.")

All cleaned CSV files loaded.


# 1. Test Final Merged Datasets

In [3]:
print("### Merging Transactions with Account Profiles...")

# Merge transactions with account profiles to enrich transaction data
# We'll merge on 'account_id'
merged_transactions_df = pd.merge(
    transactions_df_processed,
    account_profiles_df_processed,
    on='account_id',
    how='left',
    suffixes=('_txn', '_acc')
)

print("Merged DataFrame Shape:", merged_transactions_df.shape)
print("Merged DataFrame Columns:", merged_transactions_df.columns.tolist()[:10], "...") # Display first 10 columns
display(merged_transactions_df.head())
print("\nInfo of Merged DataFrame:")
merged_transactions_df.info()

### Merging Transactions with Account Profiles...
Merged DataFrame Shape: (1000000, 49)
Merged DataFrame Columns: ['transaction_id', 'account_id', 'timestamp', 'hour_of_day', 'day_of_week', 'is_weekend', 'amount', 'merchant_category', 'mcc_code', 'merchant_country'] ...


,transaction_id,account_id,timestamp,hour_of_day,day_of_week,is_weekend,amount,merchant_category,mcc_code,merchant_country,...,max_amount,fraud_count,fraud_amount,pct_foreign,avg_velocity,unique_countries,unique_categories,avg_ip_risk,fraud_rate,is_fraudster
0,TXN000000001,ACC0016173,2023-02-21 08:02:38,8,1,0,168.42,travel,4511,CA,...,885.70,0.0,0.00,0.35,1.61,6.0,12.0,27.07,0.000000,False
1,TXN000000002,ACC0011196,2024-05-12 23:13:34,23,6,1,85.78,online_retail,5999,AU,...,1928.45,0.0,0.00,0.43,0.86,7.0,9.0,24.91,0.000000,False
2,TXN000000003,ACC0001181,2023-09-22 23:28:21,23,4,0,20.15,pharmacy,5912,CA,...,660.56,0.0,0.00,0.38,0.50,5.0,9.0,21.49,0.000000,False
3,TXN000000004,ACC0037105,2022-09-28 23:26:38,23,2,0,62.49,grocery,5411,US,...,196.64,1.0,153.68,0.40,1.13,4.0,7.0,28.59,0.066667,True
4,TXN000000005,ACC0028471,2023-02-23 17:54:13,17,3,0,71.68,online_retail,5999,US,...,3353.32,0.0,0.00,0.30,1.40,4.0,6.0,21.80,0.000000,False



Info of Merged DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 49 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   transaction_id          1000000 non-null  str    
 1   account_id              1000000 non-null  str    
 2   timestamp               1000000 non-null  str    
 3   hour_of_day             1000000 non-null  int64  
 4   day_of_week             1000000 non-null  int64  
 5   is_weekend              1000000 non-null  int64  
 6   amount                  1000000 non-null  float64
 7   merchant_category       1000000 non-null  str    
 8   mcc_code                1000000 non-null  int64  
 9   merchant_country        1000000 non-null  str    
 10  card_present            1000000 non-null  bool   
 11  device_type             1000000 non-null  str    
 12  device_known            1000000 non-null  bool   
 13  ip_risk_score           1000000 non-null  

# 2. Dashboard KPI Tables

In [4]:
print("### Creating Overall Fraud Metrics KPI Table")

overall_fraud_kpis = pd.DataFrame({
    'Metric': [
        'Total Transactions',
        'Total Fraudulent Transactions',
        'Fraud Rate (%)',
        'Total Amount Transacted',
        'Total Fraud Amount',
        'Average Transaction Amount',
        'Average Fraud Amount'
    ],
    'Value': [
        len(merged_transactions_df),
        merged_transactions_df['is_fraud'].sum(),
        merged_transactions_df['is_fraud'].mean() * 100,
        merged_transactions_df['amount'].sum(),
        merged_transactions_df.loc[merged_transactions_df['is_fraud'] == True, 'amount'].sum(),
        merged_transactions_df['amount'].mean(),
        merged_transactions_df.loc[merged_transactions_df['is_fraud'] == True, 'amount'].mean()
    ]
})

display(overall_fraud_kpis)

print("\n### Creating Fraud Metrics by Merchant Category")
merchant_fraud_kpis = merged_transactions_df.groupby('merchant_category').agg(
    total_transactions=('transaction_id', 'count'),
    total_fraud_transactions=('is_fraud', 'sum'),
    total_amount=('amount', 'sum'),
    total_fraud_amount=('amount', lambda x: x[merged_transactions_df.loc[x.index, 'is_fraud']].sum())
).reset_index()
merchant_fraud_kpis['fraud_rate_pct'] = (merchant_fraud_kpis['total_fraud_transactions'] / merchant_fraud_kpis['total_transactions']) * 100
merchant_fraud_kpis = merchant_fraud_kpis.sort_values('fraud_rate_pct', ascending=False)
display(merchant_fraud_kpis.head())

print("\n### Creating Fraud Metrics by Account Type")
account_type_fraud_kpis = merged_transactions_df.groupby('account_type').agg(
    total_transactions=('transaction_id', 'count'),
    total_fraud_transactions=('is_fraud', 'sum'),
    total_amount=('amount', 'sum'),
    total_fraud_amount=('amount', lambda x: x[merged_transactions_df.loc[x.index, 'is_fraud']].sum())
).reset_index()
account_type_fraud_kpis['fraud_rate_pct'] = (account_type_fraud_kpis['total_fraud_transactions'] / account_type_fraud_kpis['total_transactions']) * 100
account_type_fraud_kpis = account_type_fraud_kpis.sort_values('fraud_rate_pct', ascending=False)
display(account_type_fraud_kpis.head())

### Creating Overall Fraud Metrics KPI Table


,Metric,Value
0,Total Transactions,1.000000e+06
1,Total Fraudulent Transactions,1.714300e+04
2,Fraud Rate (%),1.714300e+00
3,Total Amount Transacted,1.837383e+08
4,Total Fraud Amount,1.251673e+07
5,Average Transaction Amount,1.837383e+02
6,Average Fraud Amount,7.301367e+02



### Creating Fraud Metrics by Merchant Category


,merchant_category,total_transactions,total_fraud_transactions,total_amount,total_fraud_amount,fraud_rate_pct
2,crypto,19976,1067,21941685.30,2973178.82,5.341410
8,money_transfer,30079,1242,23052498.67,2526927.54,4.129127
4,gambling,9920,354,5057312.38,495852.72,3.568548
12,travel,49367,1288,27830177.78,2076275.03,2.609030
3,electronics,60229,1372,23994720.15,1554183.59,2.277972



### Creating Fraud Metrics by Account Type


,account_type,total_transactions,total_fraud_transactions,total_amount,total_fraud_amount,fraud_rate_pct
0,business,197218,3463,3.640502e+07,2415375.91,1.755925
1,personal,703360,12062,1.292753e+08,8995514.87,1.714911
2,premium,99422,1618,1.805798e+07,1105843.31,1.627406
